In [171]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu

from lifelines import KaplanMeierFitter
from lifelines import CoxPHFitter
from lifelines.statistics import logrank_test

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (make_scorer,roc_auc_score,roc_curve, precision_score, fbeta_score, f1_score, recall_score,
confusion_matrix, classification_report, accuracy_score, ConfusionMatrixDisplay)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RandomizedSearchCV

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

import warnings
warnings.filterwarnings("ignore") # suppress noncritical warnings to keep output clean

# Read Heart Failure Dataset

In [173]:
df = pd.read_csv("..\data\heart_failure_clinical_records_dataset.csv")
df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


We prepare dataset for machine learning modeling

# Define X and y

In [176]:
drop_cols = ["time", "DEATH_EVENT"] # we droped time to avoid data leakage because follow up time is part of the survival outcomes
X = df.drop(columns = drop_cols)
y = df["DEATH_EVENT"]
X.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0


In [177]:
y.head()

0    1
1    1
2    1
3    1
4    1
Name: DEATH_EVENT, dtype: int64

# Split Dataset

In [179]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Logistic Regression Model

In [181]:
# We extract numeric and categorical features
numeric_cols = ["age", "creatinine_phosphokinase", "ejection_fraction", "platelets", "serum_creatinine", "serum_sodium"]
categorical_cols = ["anaemia", "diabetes", "high_blood_pressure", "sex","smoking"]


In [182]:
#We preprocess the columns in a transformer
preprocessor = ColumnTransformer(transformers =[("num", StandardScaler(), numeric_cols),("cat", "passthrough", categorical_cols)], remainder="drop")

In [183]:
# We buid a pipeline
log_pipeline = Pipeline(steps=[("preprocessor", preprocessor),
                        ("classifier",LogisticRegression(max_iter=1000, solver="lbfgs", C = 0.1, class_weight="balanced", random_state= 42))])

In [184]:
# We fit the model
log_pipeline.fit(X_train, y_train);

We check the model performance on training set

In [186]:
y_train_pred = log_pipeline.predict(X_train)
print(classification_report(y_train, y_train_pred, target_names=["no_death_event","death_event"]))

                precision    recall  f1-score   support

no_death_event       0.88      0.78      0.83       162
   death_event       0.63      0.77      0.69        77

      accuracy                           0.78       239
     macro avg       0.75      0.78      0.76       239
  weighted avg       0.80      0.78      0.78       239



We check the model performance on test set

In [188]:
y_test_pred = log_pipeline.predict(X_test)
print(classification_report(y_test, y_test_pred, target_names=["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.84      0.78      0.81        41
   death_event       0.59      0.68      0.63        19

      accuracy                           0.75        60
     macro avg       0.72      0.73      0.72        60
  weighted avg       0.76      0.75      0.75        60



In [189]:
cm = confusion_matrix(y_test, y_test_pred)
print(cm)

[[32  9]
 [ 6 13]]


The logistic regression model correctly identified 68% of the actual death-event cases in the test set. There were 19 actual death-event cases, and the model detected approximately 13 of them. The precision for the death-event class was also 0.59, meaning that among patients predicted as death-event cases, about 59% truly experienced a death event. Although the model shows moderate ability to detect mortality cases, it still misses some of the actual death-event patients, which is an important limitation in a healthcare risk prediction setting.

## Hyperparameter Tuning

We tune the hyperparameters and employ a cross validation for possible improve performance.

In [193]:
para_grid = {"classifier__C": [0.01, 0.02, 0.03,0.04,0.05], "classifier__penalty": ["l2"], 
             "classifier__solver": ["lbfgs"], "classifier__class_weight": ["balanced"] }

In [194]:
cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state =42)
score = make_scorer(recall_score,labels =[1], average="macro")
random_search = RandomizedSearchCV(estimator=log_pipeline,
                          param_distributions= para_grid,
                          n_iter = 5,
                          cv = cv,
                          scoring=score,
                          refit = True,
                          random_state=42,
                          n_jobs=-1)

In [195]:
random_search.fit(X_train, y_train);

In [196]:
print("Best parameters:", random_search.best_params_)

Best parameters: {'classifier__solver': 'lbfgs', 'classifier__penalty': 'l2', 'classifier__class_weight': 'balanced', 'classifier__C': 0.03}


In [197]:
# We choose the best model
best_logistic_model = random_search.best_estimator_

In [198]:
# We predict using the best model
y_train_pred = best_logistic_model.predict(X_train)

In [199]:
# We check performance on training set
print(classification_report(y_train, y_train_pred, target_names=["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.87      0.80      0.83       162
   death_event       0.63      0.74      0.68        77

      accuracy                           0.78       239
     macro avg       0.75      0.77      0.76       239
  weighted avg       0.79      0.78      0.78       239



In [200]:
# We check performance on test set
y_test_pred = best_logistic_model.predict(X_test)

In [201]:
# We check the performance
print(classification_report(y_test, y_test_pred, target_names = ["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.86      0.78      0.82        41
   death_event       0.61      0.74      0.67        19

      accuracy                           0.77        60
     macro avg       0.74      0.76      0.74        60
  weighted avg       0.78      0.77      0.77        60



After applying cross-validation and hyperparameter tuning, the logistic regression model achieved a recall of 0.74 for the death-event class on the test set. This means the model correctly detected approximately 74% of the patients who actually experienced a death event. The model also achieved a precision of 0.61 for the death-event class, meaning that among patients predicted as death-event cases, 61% were truly death-event patients. Overall, the tuned model shows moderate ability to identify mortality-risk cases, with improved sensitivity to death events, although some false positives and false negatives remain. The similarity between the training and test performance suggests that the model generalizes reasonably well and does not show strong evidence of overfitting.